# Manual prompt search: 30 → 15 → 8

A cleaned version of the original experiment. Archived results are the default. Online steps are explicit and run only when `RUN_ONLINE` is set to `True`. No third aggregation or manual deletion/refill cells are included.

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "prompt_search":
    ROOT = ROOT.parent
if not (ROOT / "prompt_search" / "history").is_dir():
    raise RuntimeError("Open this notebook from the repository root or prompt_search directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from prompt_search.history import load_history, summary
RUN_ONLINE = False
summary()

## 1. Inspect the original 30 / 15 / 8 rankings

These files preserve historical prompt text, hashes, IDs and scores. They are not changed by new searches.

In [ ]:
import pandas as pd

history = load_history()
for stage, records in history.items():
    print(stage, len(records))
    display(pd.DataFrame(records)[["prompt_id", "prompt_hash", "score"]])
best = max((record for records in history.values() for record in records), key=lambda r: r["score"])
print(best["prompt_hash"], best["score"])
print(best["prompt_text"])

## 2. Configure a fresh online run

Set `DEEPSEEK_API_KEY` in the kernel environment and supply your local BiomedBERT model directory, or set `BIOMEDBERT_MODEL` before starting the kernel. The cells below create outputs only for this new run. This operation calls the paid API and trains a GCN for every evaluated candidate.

In [ ]:
if RUN_ONLINE:
    import random
    import numpy as np
    from datetime import datetime, timezone
    from dataclasses import asdict
    from data_processing.embedding import DEFAULT_EMBED_MODEL, BiomedicalEmbedder
    from prompt_search.client import make_client
    from prompt_search.evaluation import EvaluationConfig, PromptEvaluator
    from prompt_search.templates import ManualMicrobiomePrompter
    from prompt_search.search import SearchConfig, generate_initial, evaluate_initial, fuse_round

    random.seed(42)
    np.random.seed(42)
    search_config = SearchConfig()
    evaluation_config = EvaluationConfig(embed_model=DEFAULT_EMBED_MODEL, device="cuda")
    client = make_client()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run_dir = ROOT / "outputs" / "prompt_search" / stamp
    run_dir.mkdir(parents=True, exist_ok=False)
    (run_dir / "config.json").write_text(json.dumps({"search": asdict(search_config), "evaluation": asdict(evaluation_config)}, indent=2))
    evaluator = PromptEvaluator(run_dir / "evaluations", evaluation_config, client=client)
    template = ManualMicrobiomePrompter()

## 3. Generate 30 initial candidates, then evaluate them

Duplicate generations are logged without refill loops. Evaluations cache within the new run directory.

In [ ]:
if RUN_ONLINE:
    candidates, seen = generate_initial(client, template, run_dir, search_config)
    print("Unique candidates:", len(candidates))

In [ ]:
if RUN_ONLINE:
    initial_results = evaluate_initial(candidates, evaluator, run_dir)
    display(pd.DataFrame(initial_results)[["prompt_id", "prompt_hash", "score"]])

## 4. First aggregation: target 15

Rank four-parent combinations by summed pairwise BiomedBERT cosine distance. Retain new prompts only when their score exceeds the best initial score, with at most 60 combinations.

In [ ]:
if RUN_ONLINE:
    semantic_embedder = BiomedicalEmbedder(evaluation_config.embed_model, evaluation_config.device)
    round1_results, round1_status = fuse_round(
        initial_results, "round1", 4, 15, 60, 1.0,
        run_dir, evaluator, semantic_embedder, client, template, seen,
    )
    print(round1_status)
    display(pd.DataFrame(round1_results))

## 5. Second aggregation: target 8

Use three-parent combinations from round 1, with at most 70 combinations and temperature 0.5. Partial counts are reported instead of forcing a full target.

In [ ]:
if RUN_ONLINE:
    round2_results, round2_status = fuse_round(
        round1_results, "round2", 3, 8, 70, 0.5,
        run_dir, evaluator, semantic_embedder, client, template, seen,
    )
    print(round2_status)
    display(pd.DataFrame(round2_results))

## 6. Save the best prompt and actual stage counts

The historical best score is a prompt-selection score. New scores can differ because the API, text-model files, GPU computation and notebook execution order are not fixed by this archive.

In [ ]:
if RUN_ONLINE:
    all_results = [r for stage in [initial_results, round1_results, round2_results] for r in stage if r["score"] >= 0]
    payload = {"initial": {"target": 30, "actual": len(initial_results)}, "round1": round1_status, "round2": round2_status}
    if all_results:
        best_new = max(all_results, key=lambda r: r["score"])
        (run_dir / "best_prompt.txt").write_text(best_new["prompt_text"] + "\n")
        payload.update(best_prompt_hash=best_new["prompt_hash"], best_score=best_new["score"])
    (run_dir / "summary.json").write_text(json.dumps(payload, indent=2))
    print(run_dir)
    print(payload)